# Simulate a calibrant diffraction image

**Exploratory notebook.** Uses `pyFAI` to generate a synthetic AgBh calibrant
image (sharp rings at 10 keV plus a diffuse background at 30 keV) and save it as
TIFF, for testing downstream 2D processing without beamline data.

No beamline hardware required.
Edit the `# INPUT` cell before running.

In [ ]:
from pyFAI.calibrant import get_calibrant
from pyFAI.detectors import Eiger2CdTe_16M
from pyFAI.azimuthalIntegrator import AzimuthalIntegrator
from pyFAI.gui import jupyter
from scidata.io_utils import format_windows_path
import numpy as np
from PIL import Image

In [ ]:
# INPUT
output_fpath = r"path/to/output"
image_name = "SimAgBh_10keV_SD0p7m"
calibrant = "AgBh"
dist = 0.7
poni1 = 0.02
poni2 = 0.02

# fpath and image path processing
output_fpath = format_windows_path(output_fpath)
output_image_path = output_fpath + "/" + image_name + ".TIFF"

# Functions
def energy_to_wavelength(energy):
    # Input: energy in the unit of keV
    # Output: wavelength in the unit of m.
    wavelength = 12.39847/energy*1e-10
    return wavelength

def save_TIFF_image(data, output_image_path, Image_mode="I"):
    data_image = Image.fromarray(data.astype(np.int32), mode=Image_mode)
    data_image.save(output_image_path)

# Calibrant and detector definition
cal = get_calibrant(calibrant)
det = Eiger2CdTe_16M()

# ----------------------------------------------
# Sharp peak data
wavelength = energy_to_wavelength(10)
ai = AzimuthalIntegrator(detector=det, poni1=poni1, poni2=poni2, dist=dist, wavelength=wavelength)
fake_data = cal.fake_calibration_image(ai=ai, Imax=100000, U=0, V=0, W=0.0000002)
save_TIFF_image(fake_data, output_image_path)

# Background
wavelength = energy_to_wavelength(30)
ai = AzimuthalIntegrator(detector=det, poni1=poni1, poni2=poni2, dist=dist, wavelength=wavelength)
fake_data_background = cal.fake_calibration_image(ai=ai, Imax=100, U=2, V=2, W=0.01)
output_image_path = output_image_path.replace(".TIFF","_Bg.TIFF")
save_TIFF_image(fake_data_background, output_image_path)

# Combine data with background
fake_data_with_background = fake_data + fake_data_background
output_image_path = output_image_path.replace("_Bg.TIFF","_wBg.TIFF")
save_TIFF_image(fake_data_with_background, output_image_path)
# ----------------------------------------------

# Display data
jupyter.display(fake_data_with_background)